# ⚡ Async and Streaming in LangGraph

## Learning Objectives
In this notebook, you will learn:
1. **Synchronous baseline** - build a simple tool-calling agent graph to establish a point of comparison
2. **Async node execution** - convert a node function to `async`/`await` using `model.ainvoke()`
3. **Async graph invocation** - run a compiled graph end-to-end with `graph.ainvoke()`
4. **Streaming node updates** - observe each node's output as it completes with `stream_mode="updates"`
5. **Token-level streaming** - stream individual LLM tokens with `stream_mode="messages"` and assemble them into a final message with `AIMessageChunk`

## Prerequisites
- Familiarity with LangGraph's `StateGraph`, `MessagesState`, and conditional edges (see `01_Foundations`)
- Tool calling with `@tool` and `ToolNode`
- An `OPENAI_API_KEY` configured in a `.env` file at the project root

---
## 📦 Part 0: Environment Setup

Load environment variables (API keys) from the project's `.env` file before initializing any models.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API Keys from .env
# ============================================================================
from dotenv import load_dotenv

load_dotenv()
print("✅ Environment variables loaded successfully!")

---
## 🛠️ Part 1: Baseline - A Synchronous Tool-Calling Agent

Before adding async and streaming, we first build a small, fully **synchronous** tool-calling
agent. It gives us a working point of comparison: Part 2 converts this exact graph to run
asynchronously and to stream its output, without changing what the agent actually does.

### Key Concepts:
- **`MessagesState`**: A prebuilt state schema that accumulates a list of chat messages
- **`ToolNode`**: A prebuilt node that executes any tool calls found on the last AI message
- **Conditional edges**: Route to the `tools` node when the model requests a tool call, otherwise end

In [ ]:
# ============================================================================
# IMPORTS: LangGraph, LangChain Core, and Typing
# ============================================================================
from typing import Literal

from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode

from helpers import get_experientiallabs_llm

### 1.1 🔧 Define a Tool

A minimal `get_weather` tool with two hard-coded responses, just enough to demonstrate the
tool-calling loop without depending on a real weather API.

In [ ]:
# ============================================================================
# TOOL DEFINITION: get_weather
# ============================================================================
@tool
def get_weather(location: str):
    """Call to get the current weather."""
    if location.lower() in ["munich"]:
        return "It's 15 degrees Celsius and cloudy."
    else:
        return "It's 32 degrees Celsius and sunny."

### 1.2 🤖 Initialize the Model and Bind Tools

Binding the tools to the model tells it what tools are available and how to call them -
the model itself decides *when* to call `get_weather` based on the conversation.

In [ ]:
# ============================================================================
# MODEL SETUP: Bind Tools to the LLM
# ============================================================================
tools = [get_weather]
model = get_experientiallabs_llm(model_name="gpt-4o-mini").bind_tools(tools)
print("🤖 Model initialized: gpt-4o-mini (synchronous)")

### 1.3 🧩 Define Graph Nodes

- `call_model`: invokes the LLM on the current message history
- `should_continue`: routes to the `tools` node if the last AI message requested a tool call,
  otherwise routes to `END`

In [ ]:
# ============================================================================
# GRAPH NODES: call_model and should_continue
# ============================================================================
def call_model(state: MessagesState):
    messages = state["messages"]
    response = model.invoke(messages)
    return {"messages": [response]}


def should_continue(state: MessagesState) -> Literal["tools", END]:
    messages = state["messages"]
    last_message = messages[-1]
    if last_message.tool_calls:
        return "tools"
    return END

### 1.4 🏗️ Build and Compile the Graph

A `MemorySaver` checkpointer is attached so the graph remembers prior turns for a given
`thread_id` - this lets the second call below refer back to "that city" from the first call.

In [ ]:
# ============================================================================
# GRAPH CONSTRUCTION: Wire Nodes and Edges Together
# ============================================================================
workflow = StateGraph(MessagesState)
tool_node = ToolNode(tools)

workflow.add_node("agent", call_model)
workflow.add_node("tools", tool_node)

workflow.add_edge(START, "agent")
workflow.add_conditional_edges("agent", should_continue)
workflow.add_edge("tools", "agent")

graph = workflow.compile(checkpointer=MemorySaver())
print("✅ Synchronous graph compiled successfully!")

### 1.5 ▶️ Run the Graph Synchronously

Two calls on the same `thread_id` - the second one ("that city") only works because the
checkpointer persisted the first turn's messages.

In [ ]:
# ============================================================================
# SYNCHRONOUS RUN: First Turn - Ask About the Weather
# ============================================================================
graph.invoke(
    {"messages": [HumanMessage(content="How is the weather in munich?")]},
    config={"configurable": {"thread_id": 1}},
)

In [ ]:
# ============================================================================
# SYNCHRONOUS RUN: Second Turn - Follow-Up Question (Same Thread)
# ============================================================================
graph.invoke(
    {
        "messages": [
            HumanMessage(content="What would you recommend to do in that city then?")
        ]
    },
    config={"configurable": {"thread_id": 1}},
)

---
## ⚡ Part 2: Getting Production-Ready - Async and Streaming

A synchronous `graph.invoke()` blocks until the entire run finishes, and it can only serve one
request at a time per process. Real applications instead need to:
1. Handle many concurrent requests without blocking (**async**)
2. Show the user partial progress instead of a long silent wait (**streaming**)

We now rebuild the same agent with an async node, and explore two different streaming modes.

### Key Concepts:
- **`ainvoke`**: The async counterpart of `invoke` - runs the full graph without blocking the event loop
- **`stream_mode="updates"`**: Yields one dict per node, right after that node finishes
- **`stream_mode="messages"`**: Yields individual LLM tokens as `AIMessageChunk` objects, as they're generated

### 2.1 🌊 Enable Streaming on the Model and Define an Async Node

`streaming=True` lets the model emit tokens incrementally instead of returning the full
response in one shot. The node function becomes `async` and awaits `model.ainvoke()` instead
of calling `model.invoke()`.

In [ ]:
# ============================================================================
# ASYNC SETUP: Streaming-Enabled Model and Async Node
# ============================================================================
model = get_experientiallabs_llm(model_name="gpt-4o-mini", temperature=0).bind_tools(tools)


async def call_model(state: MessagesState):
    messages = state["messages"]
    response = await model.ainvoke(messages)
    return {"messages": [response]}


print("🤖 Model re-initialized: gpt-4o-mini (streaming=True, async node)")

### 2.2 🏗️ Rebuild the Graph With the Async Node

Same structure as Part 1, just with the async `call_model` node. Note the explicit
`{"tools": "tools", END: END}` mapping on the conditional edge - functionally equivalent to
the implicit version used earlier, spelled out here for clarity.

In [ ]:
# ============================================================================
# GRAPH CONSTRUCTION: Async Version
# ============================================================================
workflow = StateGraph(MessagesState)
tool_node = ToolNode(tools)

workflow.add_node("agent", call_model)
workflow.add_node("tools", tool_node)

workflow.add_edge(START, "agent")
workflow.add_conditional_edges(
    "agent",
    should_continue,
    {"tools": "tools", END: END},
)
workflow.add_edge("tools", "agent")

graph = workflow.compile(checkpointer=MemorySaver())
print("✅ Async graph compiled successfully!")

### 2.3 ▶️ Run the Graph with `ainvoke`

Functionally identical to `graph.invoke()` from Part 1, but non-blocking - this is the version
you'd call from inside an async web handler (e.g. a FastAPI endpoint).

In [ ]:
# ============================================================================
# ASYNC RUN: Full Graph Execution with ainvoke
# ============================================================================
inputs = {"messages": [HumanMessage(content="How is the weather in Munich?")]}
config = {"configurable": {"thread_id": 2}}

await graph.ainvoke(input=inputs, config=config)

### 2.4 📋 Stream Per-Node Updates with `stream_mode="updates"`

Instead of waiting for the whole run, `astream(..., stream_mode="updates")` yields a dict as
soon as each node finishes, keyed by that node's name. Useful for showing users *which step*
the agent is currently on (e.g. "calling a tool...", "generating a response...").

In [ ]:
# ============================================================================
# STREAMING: Per-Node Updates
# ============================================================================
inputs = {"messages": [HumanMessage(content="How is the weather in Munich?")]}

async for output in graph.astream(inputs, stream_mode="updates", config=config):
    # stream_mode="updates" yields dictionaries with output keyed by node name
    for key, value in output.items():
        print(f"Output from node '{key}':")
        print("---")
        value["messages"][-1].pretty_print()

### 2.5 🔍 Stream Individual Tokens with `stream_mode="messages"`

This mode yields `(message_chunk, metadata)` tuples token-by-token as the LLM generates them -
what powers a "typing" effect in a chat UI. We print each token as it arrives, and separately
accumulate the `AIMessageChunk` objects with `+` to reconstruct the final assembled message.

In [ ]:
# ============================================================================
# STREAMING: Token-Level Streaming with AIMessageChunk Assembly
# ============================================================================
from langchain_core.messages import AIMessageChunk, HumanMessage

inputs = [HumanMessage(content="How is the weather in Munich?")]
gathered = None

async for msg, metadata in graph.astream(
    {"messages": inputs}, stream_mode="messages", config=config
):
    if msg.content and not isinstance(msg, HumanMessage):
        # Print each token as it streams in
        print(msg.content, end="|", flush=True)

    # Handle the AI message chunks for proper assembly
    if isinstance(msg, AIMessageChunk):
        if gathered is None:
            gathered = msg
        else:
            gathered = gathered + msg

In [ ]:
# ============================================================================
# STREAMING: Final Assembled Message
# ============================================================================
print(gathered.content)

---
## 📝 Summary

In this notebook, we learned:

### 1. Synchronous Baseline
- **`graph.invoke()`**: Blocks until the full run completes; fine for scripts, not for
  concurrent production traffic
- **Checkpointing**: A `MemorySaver` lets multiple calls on the same `thread_id` share
  conversation history

### 2. Async Execution
- **Async nodes**: A node function can be `async def` and `await model.ainvoke(...)`
  instead of calling `model.invoke(...)`
- **`graph.ainvoke()`**: The non-blocking counterpart of `graph.invoke()` - the version to use
  inside an async web handler

### 3. Streaming
- **`stream_mode="updates"`**: Yields one output per node, right after it finishes - good for
  showing step-by-step progress
- **`stream_mode="messages"`**: Yields individual LLM tokens as `AIMessageChunk` objects -
  good for a token-by-token "typing" UI effect
- **Chunk assembly**: `AIMessageChunk` objects support `+`, letting you reconstruct the final
  complete message from the individual streamed chunks

### Next Steps
- Explore `03_Human_in_the_Loop` for pausing a running graph to collect human input mid-execution
- See `04_Advanced_State` for richer state schemas beyond `MessagesState`